# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.tooling import tool

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
chroma_client = chromadb.PersistentClient(path="chromadb")
embed_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)
collection = chroma_client.get_collection("udaplay", embedding_function=embed_fn)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [ ]:
@tool
def retrieve_game(query: str) -> list[dict]:
    """
    Semantic search: finds most results in the vector DB
    args:
    - query: a question about game industry

    Returns a list of dicts, each with Platform, Name, YearOfRelease, Description
    """
    res = collection.query(query_texts=[query], n_results=3)
    return res["metadatas"][0]

#### Evaluate Retrieval Tool

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="whether the retrieved documents are useful to answer the question")
    description: str = Field(description="explanation of the evaluation result")


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[dict]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge = LLM(model="gpt-4o-mini", temperature=0)
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query: {question}\n"
        f"Documents: {json.dumps(retrieved_docs)}"
    )
    res = judge.invoke(prompt, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(res.content)
    return report.model_dump()

#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str) -> str:
    """
    Searches the web for information about the video game industry
    args:
    - question: a question about game industry
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    res = client.search(query=question, search_depth="advanced")
    return "\n".join(f"{r['title']}: {r['content']}" for r in res["results"])

### Agent

In [ ]:
instructions = (
    "You are UdaPlay, a research agent for the video game industry.\n"
    "For every question, follow this sequence strictly:\n"
    "1. Call retrieve_game to search internal knowledge first.\n"
    "2. Call evaluate_retrieval to judge if the retrieved documents are useful.\n"
    "3. If they are not useful, call game_web_search to fill the gap.\n"
    "4. Answer the question and state whether the source was internal knowledge or a web search."
)

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0
)

In [ ]:
queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?"
]

for q in queries:
    run = agent.invoke(q)
    print(f"Q: {q}")
    print(f"A: {run.get_final_state()['messages'][-1].content}\n")

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes